In [230]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import pandas as pd
import spacy
import re
import time
import numpy as np
import time

import ast

In [231]:
from spacy.matcher import PhraseMatcher
from spacy.matcher import Matcher
import re

In [232]:
# Model data
model = {'en': spacy.load("en_core_web_sm"), 'de': spacy.load("de_core_news_sm")}

In [233]:
#function to split string in the row by " | "
def split_form(row):
    row_split = row.split(" | ")
    return row_split

In [234]:
#function to split words and sentences into two different dataframes (currently doesn't used)
def split_words_sentences_short(df):
    list_word = []
    list_sentence = []

    list_word_type_word = []
    list_word_type_sentence = []

    list_category_word = []
    list_category_sentence = []

    list_primary_subcategory_word = []
    list_primary_subcategory_sentence = []

    list_alternatives_word = []
    list_alternatives_sentence = []

    for index, row in df.iterrows():
        word = row["Lemma"]
        word_list = word.split() # Split `a_string` by whitespace.
        number_of_words = len(word_list)
        if number_of_words == 1:
            list_word.append(word)
            list_word_type_word.append(row["Word_Type"])
            list_category_word.append(row["Category"])
            list_primary_subcategory_word.append(row["Primary_subcategory"])
            list_alternatives_word.append(row["Alt_split"])
        
        
        else:
            list_sentence.append(word)
            list_word_type_sentence.append(row["Word_Type"])
            list_category_sentence.append(row["Category"])
            list_primary_subcategory_sentence.append(row["Primary_subcategory"])
            list_alternatives_sentence.append(row["Alt_split"])
        
    df_word = pd.DataFrame(list(zip(list_word, list_word_type_word, list_category_word, list_primary_subcategory_word, list_alternatives_word)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Alt_split"])
    df_sentence = pd.DataFrame(list(zip(list_sentence, list_word_type_sentence, list_category_sentence, list_primary_subcategory_sentence, list_alternatives_sentence)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Alt_split"])
    
    return df_word, df_sentence
        
    


In [235]:
#function to split words and sentences into two different dataframes for singular/plural dataframes
def split_words_sentences(df):
    list_word = []
    list_sentence = []

    list_word_type_word = []
    list_word_type_sentence = []

    list_category_word = []
    list_category_sentence = []

    list_primary_subcategory_word = []
    list_primary_subcategory_sentence = []

    list_sg_all_word = []
    list_sg_all_sentence = []
    
    list_pl_word = []
    list_pl_sentence = []

    list_pl_all_word = []
    list_pl_all_sentence = []

    list_alternatives_word = []
    list_alternatives_sentence = []
    
    for index, row in df.iterrows():
        word = row["Lemma"]
        word_list = word.split() # Split `a_string` by whitespace.
        number_of_words = len(word_list)
        if number_of_words == 1:
            list_word.append(word)
            list_word_type_word.append(row["Word_Type"])
            list_category_word.append(row["Category"])
            list_primary_subcategory_word.append(row["Primary_subcategory"])
            list_sg_all_word.append(row["Sg_all_split"])
            list_pl_word.append(row["Pl_all"])
            list_pl_all_word.append(row["Pl_all_split"])
            list_alternatives_word.append(row["Alt_split"])
        
        
        else:
            list_sentence.append(word)
            list_word_type_sentence.append(row["Word_Type"])
            list_category_sentence.append(row["Category"])
            list_primary_subcategory_sentence.append(row["Primary_subcategory"])
            list_sg_all_sentence.append(row["Sg_all_split"])
            list_pl_sentence.append(row["Pl_all"])
            list_pl_all_sentence.append(row["Pl_all_split"])
            list_alternatives_sentence.append(row["Alt_split"])
        
    df_word = pd.DataFrame(list(zip(list_word, list_word_type_word, list_category_word, list_primary_subcategory_word, list_sg_all_word, list_pl_word, list_pl_all_word, list_alternatives_word)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"])
    df_sentence = pd.DataFrame(list(zip(list_sentence, list_word_type_sentence, list_category_sentence, list_primary_subcategory_sentence, list_sg_all_sentence, list_pl_sentence, list_pl_all_sentence, list_alternatives_sentence)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"])
    
    return df_word, df_sentence
        
    


In [236]:
# read csv file
df_rules = pd.read_csv("rules_12MAY_German_h.csv")

In [237]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1860 entries, 0 to 1859
Data columns (total 16 columns):
Lemma                             1859 non-null object
Word_Type                         1859 non-null object
Priority                          280 non-null object
Category                          1859 non-null object
Primary_subcategory               1859 non-null object
Secondary_subcategory             451 non-null object
Alt_Field                         834 non-null object
Alt_Sg_Replacement                627 non-null object
Alt_Sg_/_and_inclusive_form       288 non-null object
Alt_Sg_people_first               228 non-null object
Alt_Sg_reframed                   206 non-null object
Alt_Pl_collective_noun            47 non-null object
Alt_Pl                            166 non-null object
Alt_Pl_pair_and_inclusive_form    253 non-null object
Alt_Pl_people_first               144 non-null object
3rd_party_alternatives            3 non-null object
dtypes: object(16)
memory us

In [238]:
#drop row if all value in row are missing
df_rules.dropna(how='all', inplace=True)

In [239]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1859 entries, 0 to 1859
Data columns (total 16 columns):
Lemma                             1859 non-null object
Word_Type                         1859 non-null object
Priority                          280 non-null object
Category                          1859 non-null object
Primary_subcategory               1859 non-null object
Secondary_subcategory             451 non-null object
Alt_Field                         834 non-null object
Alt_Sg_Replacement                627 non-null object
Alt_Sg_/_and_inclusive_form       288 non-null object
Alt_Sg_people_first               228 non-null object
Alt_Sg_reframed                   206 non-null object
Alt_Pl_collective_noun            47 non-null object
Alt_Pl                            166 non-null object
Alt_Pl_pair_and_inclusive_form    253 non-null object
Alt_Pl_people_first               144 non-null object
3rd_party_alternatives            3 non-null object
dtypes: object(16)
memory us

### Drop duplicates

In [240]:
df_rules[df_rules["Lemma"].duplicated(keep = False)]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_pair_and_inclusive_form,Alt_Pl_people_first,3rd_party_alternatives
13,Ally,s,12/2/2022,inclusive,d_and_i,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,Ally,s,NaN,inclusive,d_and_i,NaN,NaN,Unterstützer~in,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
609,LGBTQIA+,s,NaN,inclusive,d_and_i,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
610,LGBTQIA+,abbr,NaN,style,abbreviation,NaN,NaN,"lesbian, gay, bisexual, transgender, intersex,...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1116,Zwischengeschlechtlichkeit,s,12/2/2022,unconscious_bias,gender_identity,NaN,NaN,Intergeschlechtlichkeit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1117,Zwischengeschlechtlichkeit,s,NaN,unconscious_bias,gender_identity,NaN,NaN,Intergeschlechtlichkeit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [241]:
#get index of duplicates
i1 = df_rules[((df_rules.Lemma == 'Ally'))].index

In [242]:
#get index of duplicates
i2 = df_rules[((df_rules.Lemma == 'LGBTQIA+'))].index

In [243]:
#get index of duplicates
i3 = df_rules[((df_rules.Lemma == 'Zwischengeschlechtlichkeit'))].index

In [244]:
#drop row with one duplicate
df_rules = df_rules.drop(i1[1])

In [245]:
#drop row with one duplicate
df_rules = df_rules.drop(i2[0])

In [246]:
#drop row with one duplicate
df_rules = df_rules.drop(i3[0])

In [247]:
df_rules["Lemma"].duplicated().sum()

0

In [248]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1856 entries, 0 to 1859
Data columns (total 16 columns):
Lemma                             1856 non-null object
Word_Type                         1856 non-null object
Priority                          279 non-null object
Category                          1856 non-null object
Primary_subcategory               1856 non-null object
Secondary_subcategory             451 non-null object
Alt_Field                         834 non-null object
Alt_Sg_Replacement                625 non-null object
Alt_Sg_/_and_inclusive_form       288 non-null object
Alt_Sg_people_first               228 non-null object
Alt_Sg_reframed                   206 non-null object
Alt_Pl_collective_noun            47 non-null object
Alt_Pl                            166 non-null object
Alt_Pl_pair_and_inclusive_form    253 non-null object
Alt_Pl_people_first               144 non-null object
3rd_party_alternatives            3 non-null object
dtypes: object(16)
memory us

### Check if subcategory name is missing and set the parent category name to subcategory if the subategory name is missing

In [249]:
#check missing values in Primary subcategory
df_rules[df_rules['Primary_subcategory'].isna()]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_pair_and_inclusive_form,Alt_Pl_people_first,3rd_party_alternatives


In [250]:
#fill the missing values in "primary subcategory" by values from "Category"
df_rules.Primary_subcategory.fillna(df_rules.Category, inplace=True)

In [251]:
#check if the values are filled
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1856 entries, 0 to 1859
Data columns (total 16 columns):
Lemma                             1856 non-null object
Word_Type                         1856 non-null object
Priority                          279 non-null object
Category                          1856 non-null object
Primary_subcategory               1856 non-null object
Secondary_subcategory             451 non-null object
Alt_Field                         834 non-null object
Alt_Sg_Replacement                625 non-null object
Alt_Sg_/_and_inclusive_form       288 non-null object
Alt_Sg_people_first               228 non-null object
Alt_Sg_reframed                   206 non-null object
Alt_Pl_collective_noun            47 non-null object
Alt_Pl                            166 non-null object
Alt_Pl_pair_and_inclusive_form    253 non-null object
Alt_Pl_people_first               144 non-null object
3rd_party_alternatives            3 non-null object
dtypes: object(16)
memory us

## Alternatives Single

### Single click+rephrase. Important order of columns with alternatives

In [252]:
#order for plural form of alternatives provided by Tracey
df_rules["Sg_all"]= df_rules.loc[:, ("Alt_Sg_Replacement", "Alt_Field", "Alt_Sg_/_and_inclusive_form", "Alt_Pl_collective_noun", "Alt_Pl", "Alt_Sg_people_first", "Alt_Sg_reframed")].apply(
    lambda x: " | ".join(x.dropna().astype(str)), axis=1)

In [253]:
# split alternatives in the row by " | " applying function split_form to convert string alternatives into array
df_rules["Sg_all_split"]=df_rules["Sg_all"].map(split_form)

## Alternatives Plural

### Single click + rephrase. Important order of columns with alternatives

In [254]:
#order for plural form of alternatives provided by Tracey
df_rules["Pl_all"]= df_rules.loc[:, ("Alt_Pl", "Alt_Pl_pair_and_inclusive_form", "Alt_Pl_collective_noun", "Alt_Field", "Alt_Pl_people_first")].apply(
    lambda x: " | ".join(x.dropna().astype(str)), axis=1)

In [255]:
# split alternatives in the row by " | " applying function split_form
df_rules["Pl_all_split"]=df_rules["Pl_all"].map(split_form)

### Merge alternatives into one column

In [256]:
source_col_loc = df_rules.columns.get_loc('Alt_Field') # column position starts from 0

In [257]:
#merge alternatives in one column by sign  " | "
df_rules['alternatives'] = df_rules.iloc[:,source_col_loc:source_col_loc+9].apply(
    lambda x: " | ".join(x.dropna().astype(str)), axis=1)


In [258]:
# split alternatives in the row by " | " applying function split_form
df_rules["Alt_split"]=df_rules["alternatives"].map(split_form)

In [259]:
df_rules.head()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Alt_Sg_people_first,...,Alt_Pl,Alt_Pl_pair_and_inclusive_form,Alt_Pl_people_first,3rd_party_alternatives,Sg_all,Sg_all_split,Pl_all,Pl_all_split,alternatives,Alt_split
0,Abdichter,s,NaN,gendered,titles,NaN,Abdichtung,NaN,Abdichter~in~/~Abdichter,NaN,...,NaN,Abdichter~innen~ und ~Abdichter,NaN,NaN,Abdichtung | Abdichter~in~/~Abdichter,"[Abdichtung, Abdichter~in~/~Abdichter]",Abdichter~innen~ und ~Abdichter | Abdichtung,"[Abdichter~innen~ und ~Abdichter, Abdichtung]",Abdichtung | Abdichter~in~/~Abdichter | Abdich...,"[Abdichtung, Abdichter~in~/~Abdichter, Abdicht..."
1,Abgeordneter,s,26-01-2022,gendered,titles,NaN,NaN,NaN,Abgeordnet~e~/~Abgeordneter,NaN,...,Abgeordnete,NaN,NaN,NaN,Abgeordnet~e~/~Abgeordneter | Abgeordnete,"[Abgeordnet~e~/~Abgeordneter, Abgeordnete]",Abgeordnete,[Abgeordnete],Abgeordnet~e~/~Abgeordneter | Abgeordnete,"[Abgeordnet~e~/~Abgeordneter, Abgeordnete]"
2,Abhängigkeit,s,NaN,inclusive,communal,caring,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,,[],,[],,[]
3,Abwichshure,s,NaN,openly_discriminating,sexism,NaN,-,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,-,[-],-,[-],-,[-]
4,Achtsamkeit,s,NaN,inclusive,communal,diligence,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,,[],,[],,[]


### Create subsets of dataframes

In [260]:
df_rules = df_rules[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"]]

### Check duplicates&homonyms

In [261]:
df_rules[df_rules["Lemma"].duplicated(keep = False)]

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split


## Acronyms (abbreviations)

In [262]:
df_rules["Word_Type"].unique()

array(['s', 'S', 'i', 'abbr', 'a', 's+v', 'acr', 'v', 'adv+a', 'v+s',
       'adv', 'a+s', 'a+v'], dtype=object)

In [263]:
df_abbr = df_rules[(df_rules["Word_Type"]=='abbr')]

In [264]:
df_abbr.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 11 entries, 179 to 1721
Data columns (total 8 columns):
Lemma                  11 non-null object
Word_Type              11 non-null object
Category               11 non-null object
Primary_subcategory    11 non-null object
Sg_all_split           11 non-null object
Pl_all                 11 non-null object
Pl_all_split           11 non-null object
Alt_split              11 non-null object
dtypes: object(8)
memory usage: 792.0+ bytes


In [265]:
#create subset of dataframe to drop unnecessary columns
df_abbr = df_abbr[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [266]:
#save acronyms (abbreviations) to csv file after preprocessing
df_abbr.to_csv("abbreviations_DE.csv", index=False)

### Remove acronymes (abbr) from dataframe for the next steps pre-processing

In [ ]:
#remove abbr
abbr_list = list(df_abbr["Lemma"])

In [ ]:
df_rules = df_rules.query("Lemma != @abbr_list")

## Openly_discriminating

In [267]:
df_openly_discrim = df_rules[df_rules["Category"]=="openly_discriminating"]

In [268]:
#info open_discrim
df_openly_discrim.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 384 entries, 3 to 1797
Data columns (total 8 columns):
Lemma                  384 non-null object
Word_Type              384 non-null object
Category               384 non-null object
Primary_subcategory    384 non-null object
Sg_all_split           384 non-null object
Pl_all                 384 non-null object
Pl_all_split           384 non-null object
Alt_split              384 non-null object
dtypes: object(8)
memory usage: 27.0+ KB


In [269]:
df_openly_discrim.head()

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
3,Abwichshure,s,openly_discriminating,sexism,[-],-,[-],[-]
6,Affenfotze,s,openly_discriminating,sexism,[-],-,[-],[-]
8,Albino,s,openly_discriminating,ableism,[-],-,[-],[-]
9,Allerweltshure,s,openly_discriminating,sexism,[-],-,[-],[-]
11,Allmannshure,s,openly_discriminating,sexism,[-],-,[-],[-]


In [270]:
#split dataframe into two words and sentences using function split_words_sentences
df_open_dis_word, df_open_dis_sentence = split_words_sentences(df_openly_discrim)

In [271]:
df_open_dis_word.head()

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
0,Abwichshure,s,openly_discriminating,sexism,[-],-,[-],[-]
1,Affenfotze,s,openly_discriminating,sexism,[-],-,[-],[-]
2,Albino,s,openly_discriminating,ableism,[-],-,[-],[-]
3,Allerweltshure,s,openly_discriminating,sexism,[-],-,[-],[-]
4,Allmannshure,s,openly_discriminating,sexism,[-],-,[-],[-]


In [272]:
#create subset of dataframe to drop unnecessary columns
df_open_dis_word = df_open_dis_word[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]
df_open_dis_sentence = df_open_dis_sentence[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [273]:
#save to csv file after preprocessing
df_open_dis_word.to_csv("open_dis_words_DE.csv", index=False)

In [274]:
#save to csv file after preprocessing
df_open_dis_sentence.to_csv("open_dis_sentences_DE.csv", index=False)

## Style

In [275]:
df_style = df_rules[df_rules["Category"]=="style"]

In [276]:
#info style
df_style.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 425 entries, 15 to 1859
Data columns (total 8 columns):
Lemma                  425 non-null object
Word_Type              425 non-null object
Category               425 non-null object
Primary_subcategory    425 non-null object
Sg_all_split           425 non-null object
Pl_all                 425 non-null object
Pl_all_split           425 non-null object
Alt_split              425 non-null object
dtypes: object(8)
memory usage: 29.9+ KB


In [277]:
#split dataframe into two words and sentences using function split_words_sentences
df_style_word, df_style_sentence = split_words_sentences(df_style)

In [278]:
#create subset of dataframe to drop unnecessary columns
df_style_word = df_style_word[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]
df_style_sentence = df_style_sentence[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [279]:
#save to csv file after preprocessing
df_style_word.to_csv("style_words_DE.csv", index=False)

In [280]:
#save to csv file after preprocessing
df_style_sentence.to_csv("style_sentences_DE.csv", index=False)

## Inclusive

In [281]:
df_inclusive = df_rules[df_rules["Category"]=="inclusive"]

In [282]:
#info inclusive
df_inclusive.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 322 entries, 2 to 1842
Data columns (total 8 columns):
Lemma                  322 non-null object
Word_Type              322 non-null object
Category               322 non-null object
Primary_subcategory    322 non-null object
Sg_all_split           322 non-null object
Pl_all                 322 non-null object
Pl_all_split           322 non-null object
Alt_split              322 non-null object
dtypes: object(8)
memory usage: 22.6+ KB


In [283]:
df_inclusive.head()

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
2,Abhängigkeit,s,inclusive,communal,[],,[],[]
4,Achtsamkeit,s,inclusive,communal,[],,[],[]
10,Allies,s,inclusive,d_and_i,[Unterstützer~innen~/~Unterstützer],,[],[Unterstützer~innen~/~Unterstützer]
12,Allonormativität,s,inclusive,d_and_i,[],,[],[]
13,Ally,s,inclusive,d_and_i,[],,[],[]


In [284]:
df_inclusive["Primary_subcategory"].unique()

array(['communal', 'd_and_i'], dtype=object)

In [285]:
#split dataframe into two words and sentences using function split_words_sentences
df_inclusive_word, df_inclusive_sentence = split_words_sentences(df_inclusive)

In [286]:
#create subset of dataframe to drop unnecessary columns
df_inclusive_word = df_inclusive_word[['Lemma', "Word_Type", "Category", "Primary_subcategory"]]
df_inclusive_sentence = df_inclusive_sentence[['Lemma', "Word_Type", "Category", "Primary_subcategory"]]

In [287]:
df_inclusive_sentence.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 4 columns):
Lemma                  17 non-null object
Word_Type              17 non-null object
Category               17 non-null object
Primary_subcategory    17 non-null object
dtypes: object(4)
memory usage: 672.0+ bytes


In [288]:
df_inclusive_word.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 305 entries, 0 to 304
Data columns (total 4 columns):
Lemma                  305 non-null object
Word_Type              305 non-null object
Category               305 non-null object
Primary_subcategory    305 non-null object
dtypes: object(4)
memory usage: 9.7+ KB


In [289]:
df_communal_word = df_inclusive_word[df_inclusive_word["Primary_subcategory"]=='communal']

In [290]:
df_communal_word.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 217 entries, 0 to 304
Data columns (total 4 columns):
Lemma                  217 non-null object
Word_Type              217 non-null object
Category               217 non-null object
Primary_subcategory    217 non-null object
dtypes: object(4)
memory usage: 8.5+ KB


In [291]:
df_communal_sentence = df_inclusive_sentence[df_inclusive_sentence["Primary_subcategory"]=='communal']

In [292]:
df_communal_sentence.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 0 entries
Data columns (total 4 columns):
Lemma                  0 non-null object
Word_Type              0 non-null object
Category               0 non-null object
Primary_subcategory    0 non-null object
dtypes: object(4)
memory usage: 0.0+ bytes


In [293]:
#save to csv file after preprocessing
df_communal_word.to_csv("communal_DE.csv", index=False)

In [294]:
df_d_and_i_word = df_inclusive_word[df_inclusive_word["Primary_subcategory"]=='d_and_i']

In [295]:
df_d_and_i_word.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 88 entries, 2 to 287
Data columns (total 4 columns):
Lemma                  88 non-null object
Word_Type              88 non-null object
Category               88 non-null object
Primary_subcategory    88 non-null object
dtypes: object(4)
memory usage: 3.4+ KB


In [296]:
df_d_and_i_sentence = df_inclusive_sentence[df_inclusive_sentence["Primary_subcategory"]=='d_and_i']

In [297]:
df_d_and_i_sentence.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 17 entries, 0 to 16
Data columns (total 4 columns):
Lemma                  17 non-null object
Word_Type              17 non-null object
Category               17 non-null object
Primary_subcategory    17 non-null object
dtypes: object(4)
memory usage: 680.0+ bytes


In [298]:
#save to csv file after preprocessing
df_d_and_i_word.to_csv("d_and_i_words_DE.csv", index=False)

In [299]:
#save to csv file after preprocessing
df_d_and_i_sentence.to_csv("d_and_i_sentences_DE.csv", index=False)

## Gendered

In [300]:
df_gendered = df_rules[df_rules["Category"]=="gendered"]

In [301]:
#info gendered
df_gendered.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 294 entries, 0 to 1845
Data columns (total 8 columns):
Lemma                  294 non-null object
Word_Type              294 non-null object
Category               294 non-null object
Primary_subcategory    294 non-null object
Sg_all_split           294 non-null object
Pl_all                 294 non-null object
Pl_all_split           294 non-null object
Alt_split              294 non-null object
dtypes: object(8)
memory usage: 20.7+ KB


In [302]:
#split dataframe into two words and sentences using function split_words_sentences
df_gendered_word, df_gendered_sentence = split_words_sentences(df_gendered)

In [303]:
df_gendered_word.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 276 entries, 0 to 275
Data columns (total 8 columns):
Lemma                  276 non-null object
Word_Type              276 non-null object
Category               276 non-null object
Primary_subcategory    276 non-null object
Sg_all_split           276 non-null object
Pl_all                 276 non-null object
Pl_all_split           276 non-null object
Alt_split              276 non-null object
dtypes: object(8)
memory usage: 17.4+ KB


In [304]:
df_gendered_sentence.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 8 columns):
Lemma                  18 non-null object
Word_Type              18 non-null object
Category               18 non-null object
Primary_subcategory    18 non-null object
Sg_all_split           18 non-null object
Pl_all                 18 non-null object
Pl_all_split           18 non-null object
Alt_split              18 non-null object
dtypes: object(8)
memory usage: 1.2+ KB


In [305]:
df_gendered_word_noun = df_gendered_word.query("Word_Type == 's'")

In [306]:
df_gendered_word_noun.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 269 entries, 0 to 269
Data columns (total 8 columns):
Lemma                  269 non-null object
Word_Type              269 non-null object
Category               269 non-null object
Primary_subcategory    269 non-null object
Sg_all_split           269 non-null object
Pl_all                 269 non-null object
Pl_all_split           269 non-null object
Alt_split              269 non-null object
dtypes: object(8)
memory usage: 18.9+ KB


In [307]:
df_gendered_word_no_noun = df_gendered_word.query("Word_Type != 's'")

In [308]:
#drop unnecessary column "Pl_all"
df_gendered_word_noun = df_gendered_word_noun.drop(labels=["Pl_all"], axis=1)
df_gendered_word_no_noun = df_gendered_word_no_noun.drop(labels=["Pl_all"], axis=1)
df_gendered_sentence = df_gendered_sentence.drop(labels=["Pl_all"], axis=1)

In [309]:
#save to csv file after preprocessing
df_gendered_word_noun.to_csv("gendered_noun_words_DE.csv", index=False)

In [310]:
#save to csv file after preprocessing
df_gendered_word_no_noun.to_csv("gendered_no_noun_words_DE.csv", index=False)

In [311]:
df_gendered_sentence.to_csv("gendered_sentences_DE.csv", index=False)

## Unconscious Bias

In [312]:
df_ub = df_rules[df_rules["Category"]=='unconscious_bias']

In [313]:
#info unc_bias
df_ub.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 431 entries, 34 to 1858
Data columns (total 8 columns):
Lemma                  431 non-null object
Word_Type              431 non-null object
Category               431 non-null object
Primary_subcategory    431 non-null object
Sg_all_split           431 non-null object
Pl_all                 431 non-null object
Pl_all_split           431 non-null object
Alt_split              431 non-null object
dtypes: object(8)
memory usage: 30.3+ KB


In [314]:
#split dataframe into two words and sentences using function split_words_sentences
df_ub_word, df_ub_sentence = split_words_sentences(df_ub)

In [315]:
df_ub_word.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 359 entries, 0 to 358
Data columns (total 8 columns):
Lemma                  359 non-null object
Word_Type              359 non-null object
Category               359 non-null object
Primary_subcategory    359 non-null object
Sg_all_split           359 non-null object
Pl_all                 359 non-null object
Pl_all_split           359 non-null object
Alt_split              359 non-null object
dtypes: object(8)
memory usage: 22.6+ KB


In [316]:
#filter datframe to have plural for subj(noun) without special caracters and empty strings
df_ub_word_pl = df_ub_word[(df_ub_word["Pl_all"]!='')&(df_ub_word["Pl_all"]!='-')&(df_ub_word["Word_Type"]=='s')]

In [317]:
df_ub_word_pl.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 36 entries, 0 to 181
Data columns (total 8 columns):
Lemma                  36 non-null object
Word_Type              36 non-null object
Category               36 non-null object
Primary_subcategory    36 non-null object
Sg_all_split           36 non-null object
Pl_all                 36 non-null object
Pl_all_split           36 non-null object
Alt_split              36 non-null object
dtypes: object(8)
memory usage: 2.5+ KB


#### save the left data into separate dataframe

In [318]:
plural_list = list(df_ub_word_pl["Lemma"])

In [319]:
df_ub_word_left = df_ub_word.query("Lemma != @plural_list")

#### remove unnecessary column "Pl_all" 

In [320]:
df_ub_word_left = df_ub_word_left.drop(labels=["Pl_all"], axis=1)
df_ub_word_pl = df_ub_word_pl.drop(labels=["Pl_all"], axis=1)
df_ub_sentence = df_ub_sentence.drop(labels=["Pl_all"], axis=1)

#### save to csv files after preprocessing

In [321]:
#save to csv files after preprocessing
df_ub_word_left.to_csv("ub_no_plur_words_DE.csv", index=False)
df_ub_word_pl.to_csv("ub_plur_words_DE.csv", index=False)
df_ub_sentence.to_csv("ub_sentences_DE.csv", index=False)